In [14]:
import pandas as pd
import numpy as np
import glicko2
import copy

#from fighters_fights.csv create a dataframe with fighter, date, glicko rating

folder = '/Users/alejandrogomez-paz/Desktop/UFC Project/2. data_cleaning/'
df = pd.read_csv(folder + 'fighters_fights.csv')
df['opponent_id'] = df.groupby('fight_id')['fighter_id'].transform(lambda x: x[::-1].values)
df = df.sort_values('date', ascending=True).drop_duplicates('fight_id').reset_index(drop=True)

rating_dict = {} # [fighter, date]: glicko rating 
initialized_fighters = set()

for i, row in df.iterrows():
    fighter, opponent, date = row['fighter_name'], row['opponent_name'], row['date']
    if fighter not in initialized_fighters:
        rating_dict[(fighter, date)] = glicko2.Player()
        initialized_fighters.add(fighter)
    if opponent not in initialized_fighters:
        rating_dict[(opponent, date)] = glicko2.Player()
        initialized_fighters.add(opponent)

    last_date = max([d for (f, d) in rating_dict.keys() if f == fighter])
    opp_last_date = max([d for (f, d) in rating_dict.keys() if f == opponent])
    fighter_object = copy.deepcopy(rating_dict[(fighter, last_date)])
    opponent_object = copy.deepcopy(rating_dict[(opponent, opp_last_date)])

    # next two lines in order to freeze variables form objects; makes logic less messy and less variable tracking :)
    fighter_rating, fighter_rd = rating_dict[(fighter, last_date)].rating, rating_dict[(fighter, last_date)].rd
    opponent_rating, opponent_rd = rating_dict[(opponent, opp_last_date)].rating, rating_dict[(opponent, opp_last_date)].rd

    if fighter == row['winner_name']:
        fighter_object.update_player([opponent_rating], [opponent_rd], [1])
        rating_dict[(fighter, date)] = fighter_object
        
        opponent_object.update_player([fighter_rating], [fighter_rd], [0])
        rating_dict[(opponent, date)] = opponent_object
    else:
        fighter_object.update_player([opponent_rating], [opponent_rd], [0])
        rating_dict[(fighter, date)] = fighter_object
        
        opponent_object.update_player([fighter_rating], [fighter_rd], [1])
        rating_dict[(opponent, date)] = opponent_object


df_ratings = pd.DataFrame(
    [(f, d, p.rating, p.rd, p.vol) for (f, d), p in rating_dict.items()],
    columns=['fighter', 'prior_to_date', 'rating', 'rating_deviation', 'volatility'])

df_ratings = df_ratings.sort_values(['fighter', 'prior_to_date'], ascending = True)

# pre-fight rating: the rating carried INTO each fight (previous fight's result)
df_ratings['rating'] = df_ratings.groupby('fighter')['rating'].shift(1).fillna(1500)
df_ratings['rating_deviation']     = df_ratings.groupby('fighter')['rating_deviation'].shift(1).fillna(350)
df_ratings['volatility'] = df_ratings.groupby('fighter')['volatility'].shift(1).fillna(0.06)

df_ratings.to_csv('ratings.csv', index = False)

In [9]:
df_ratings.head()

,fighter,prior_to_date,rating,rating_deviation,volatility
18497,AJ Cunningham,2023-09-12,1500.000000,350.000000,0.060000
18953,AJ Cunningham,2024-03-02,1331.061807,275.861300,0.060000
20165,AJ Cunningham,2025-03-15,1299.851328,254.854986,0.059999
16084,AJ Dobson,2021-09-21,1500.000000,350.000000,0.060000
16517,AJ Dobson,2022-02-12,1662.310895,290.318965,0.060000
